In [ ]:
import os
import sys
import re
import json
import time
import shutil
import hashlib
import subprocess
from datetime import datetime, timezone

In [ ]:

import os, json
INPUT_ROOT = "/kaggle/input"
RESTORE_PATH = ""        

def _audio_ft_on(d):
    try:
        return json.load(open(os.path.join(d, "features", "audio_ft_meta.json"))).get("enabled") is True
    except Exception:
        return False

def _is_anchor(d):
    """Outputs of the anchor-fusion test (Version B) are not the base for this run."""
    try:
        return json.load(open(os.path.join(d, "results", "phase9_config.json"))).get("family") == "anchor"
    except Exception:
        return False

def find_5a_output(root):
    """Folders that look like a finished Version-5a working directory (config.json + features/ built WITH dialogue context)."""
    hits = []
    for dp, dn, fn in os.walk(root):
        if dp[len(root):].count(os.sep) > 5:
            dn[:] = []
            continue
        if "config.json" in fn and os.path.isdir(os.path.join(dp, "features")):
            try:
                ok = int(json.load(open(os.path.join(dp, "features", "text_ft_meta.json"))).get("settings", {}).get("context_utterances", 0)) >= 1 and json.load(open(os.path.join(dp, "features", "text_ft_meta.json"))).get("settings", {}).get("ft_model") == "roberta-large" and not _is_anchor(dp) and _audio_ft_on(dp)
            except Exception:
                ok = False
            if ok:
                hits.append(dp)
    def _has_video_ft(d):                      
        try:
            return json.load(open(os.path.join(d, "features", "video_ft_meta.json"))).get("enabled") is True
        except Exception:
            return False
    def _is_E(d):                               
        try:
            return json.load(open(os.path.join(d, "results", "phase9_config.json"))).get("select_metric") == "accuracy" and os.path.exists(os.path.join(d, "results", "final_report.md"))
        except Exception:
            return False
    hits = [h for h in hits if not _has_video_ft(h)]
    def _newest(d):                             
        p9 = os.path.join(d, "results", "phase9_config.json")
        if _is_E(d):
            return 1e12 + os.path.getmtime(p9)
        fs = [os.path.getmtime(os.path.join(dp2, f)) for dp2, _, fn2 in os.walk(os.path.join(d, "features", "_p4b_folds")) for f in fn2]
        return max(fs) if fs else 0.0
    return sorted(hits, key=_newest, reverse=True)

if os.path.exists("/kaggle/working/config.json"):
    print("[versionE] /kaggle/working already holds a pipeline (same session): no restore needed.")
elif RESTORE_PATH:
    os.environ["MELD_RESTORE_FROM"] = RESTORE_PATH
    print("[versionE] restoring from", RESTORE_PATH)
else:
    _hits = find_5a_output(INPUT_ROOT) if os.path.isdir(INPUT_ROOT) else []
    if _hits:
        os.environ["MELD_RESTORE_FROM"] = _hits[0]
        print("[versionE] found your Version D output:", _hits[0], "" if len(_hits) == 1 else f"(also found {len(_hits) - 1} more; set RESTORE_PATH above to choose)")
    else:
        print("[versionE] WARNING: no Version D output found under /kaggle/input. The pipeline will run from the beginning (about 5 hours). "
              "To save time, add your Version 5a Kaggle output as an input dataset and run again.")

os.environ.update({
    "MELD_AUDIO_FT": "1",              
    "MELD_P3B_MODEL": "roberta-large",   
    "MELD_TEXT_CTX": "3",            
    "MELD_LOSS_SWEEP": "1",           
    "MELD_DEMO_ASSETS": "1",          
    "MELD_SELECT_METRIC": "accuracy",  
})
print("[versionE] settings applied:", {k: os.environ[k] for k in ("MELD_AUDIO_FT", "MELD_P3B_MODEL", "MELD_TEXT_CTX", "MELD_LOSS_SWEEP", "MELD_SELECT_METRIC")})


In [ ]:

def env_bool(name, default):
    v = os.environ.get(name)
    return default if v is None else v.strip().lower() in ("1", "true", "yes", "y")

BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
NOTEBOOK_DIR = os.environ.get("MELD_NOTEBOOK_DIR", "")           
START_AT = os.environ.get("MELD_START_AT", "")                   
STOP_AFTER = os.environ.get("MELD_STOP_AFTER", "")                
FORCE = [x.strip() for x in os.environ.get("MELD_FORCE", "").split(",") if x.strip()]   
ADOPT_EXISTING = env_bool("MELD_ADOPT_EXISTING", True)          
CASCADE = env_bool("MELD_CASCADE", True)                        
DRY_RUN = env_bool("MELD_DRY_RUN", False)                        
RESTORE_FROM = os.environ.get("MELD_RESTORE_FROM", "")           
os.environ["MELD_BASE_DIR"] = BASE_DIR                           
STATE_PATH = os.path.join(BASE_DIR, "pipeline_state.json")
EXEC_DIR, LOG_DIR = os.path.join(BASE_DIR, "executed_notebooks"), os.path.join(BASE_DIR, "logs")
print(f"BASE_DIR={BASE_DIR} | START_AT={START_AT or '-'} STOP_AFTER={STOP_AFTER or '-'} FORCE={FORCE or '-'} "
      f"ADOPT_EXISTING={ADOPT_EXISTING} CASCADE={CASCADE} DRY_RUN={DRY_RUN}")

In [ ]:

S = ["train", "dev", "test"]
def per(pattern, folder):
    return [f"{folder}/{pattern.format(s=s)}" for s in S]

PHASES = [
    dict(id="Phase_01", nb="Phase_01_Environment_Setup.ipynb", artifacts=["config.json", "results/environment.json"], custom="features_config_current"),
    dict(id="Phase_02", nb="Phase_02_Dataset_Preparation.ipynb",
         artifacts=per("{s}_sent_emo.csv", "data") + per("media_manifest_{s}.csv", "data")
         + ["data/dataset_summary.json", "results/leakage_report.json", "results/class_distribution.csv"]),
    dict(id="Phase_03", nb="Phase_03_Text_Feature_Extraction.ipynb",
         artifacts=per("text_features_{s}.npy", "features") + ["features/text_features_meta.json"],
         verify=["results/text_features_verification.json"]),
   
    dict(id="Phase_03b", nb="Phase_03b_Finetuned_Text_OOF.ipynb",
         artifacts=["features/text_ft_meta.json", "results/text_ft_report.json"], verify=["results/text_ft_verification.json"], custom="text_ft_current"),
    dict(id="Phase_04", nb="Phase_04_Audio_Feature_Extraction.ipynb", independent_of=["Phase_03b"],
         artifacts=per("audio_features_{s}.npy", "features") + per("audio_valid_{s}.npy", "features") + per("audio_layers_{s}.npy", "features")
         + ["features/audio_scaler.npz", "features/audio_features_meta.json"]),   
   
    dict(id="Phase_04b", nb="Phase_04b_Finetuned_Audio_OOF.ipynb", independent_of=["Phase_03b"],
         artifacts=["features/audio_ft_meta.json", "results/audio_ft_report.json"], verify=["results/audio_ft_verification.json"], custom="audio_ft_current"),
    dict(id="Phase_05", nb="Phase_05_Video_Feature_Extraction.ipynb", independent_of=["Phase_03b", "Phase_04b"],
         artifacts=per("video_features_{s}.npy", "features") + per("video_valid_{s}.npy", "features") + per("video_features_raw_{s}.npy", "features")
         + ["features/video_features_meta.json"]),  
    dict(id="Phase_06", nb="Phase_06_Context_Modeling.ipynb",
         artifacts=["meld_context.py", "context_config.json"] + per("context_manifest_{s}.csv", "data"),
         verify=["results/context_verification.json"]),
    dict(id="Phase_07", nb="Phase_07_Baseline_Fusion.ipynb", artifacts=["meld_models.py", "model_config.json"],
         verify=["results/baseline_verification.json"]),
    dict(id="Phase_08", nb="Phase_08_Proposed_Model.ipynb",
         artifacts=["results/proposed_model_summary.json", "results/reliability_scale_diagnostics.json"],
         verify=["results/proposed_verification.json"]),
    dict(id="Phase_09a", nb="Phase_09a_Loss_Sweep.ipynb", artifacts=["results/loss_sweep_selected.json"], custom="loss_sweep_current"),
    dict(id="Phase_09", nb="Phase_09_Model_Training.ipynb",
         artifacts=["meld_train.py", "results/phase9_run_index.json", "results/phase9_selected_lambda.json", "results/phase9_val_summary.csv"],
         custom="phase9_runs_complete"),
    dict(id="Phase_10", nb="Phase_10_Evaluation.ipynb", artifacts=["meld_eval.py", "results/phase10_test_results.json"],
         verify=["results/evaluation_verification.json"]),
    dict(id="Phase_10b", nb="Phase_10b_Seed_Ensemble.ipynb", optional=True,
         artifacts=["results/ensemble/ensemble_results.json", "results/ensemble/ensemble_summary.csv", "results/ensemble/ensemble_per_class.csv",
                    "results/ensemble/ensemble_size_curve.csv"],
         verify=["results/ensemble/ensemble_verification.json"]),
    dict(id="Phase_11", nb="Phase_11_Experiments_and_Analysis.ipynb", artifacts=["results/phase11_results.json"],
         verify=["results/robustness_verification.json"]),
    dict(id="Phase_12", nb="Phase_12_Final_Results.ipynb", artifacts=["results/final_report.md", "results/final_thesis_report.json"],
         verify=["results/report_verification.json"]),
    dict(id="Phase_13", nb="Phase_13_Demo_Assets.ipynb", optional=True,
         artifacts=["demo_assets/demo_manifest.json", "demo_assets/text_encoder.pt", "demo_assets/audio_encoder.pt"], verify=["results/demo_verification.json"]),
]
print(" -> ".join(p["id"] for p in PHASES))

In [ ]:
def path_in_base(rel):
    return os.path.join(BASE_DIR, rel)

def phase9_runs_complete():
    """Phase 9 is complete only if every run listed in its index finished (result.json with completed=True)."""
    idx = json.load(open(path_in_base("results/phase9_run_index.json")))
    problems = []
    for r in idx:
        res = os.path.join(r["run_dir"], "result.json")
        if not os.path.exists(res) or not json.load(open(res)).get("completed"):
            problems.append(f"unfinished run {r['name']}/seed{r['seed']}")
    return problems

def features_config_current():
    """STEP 3: an old config.json (eGeMAPS / DenseNet) must never be 'adopted' as valid for the face + WavLM pipeline."""
    cfg = json.load(open(path_in_base("config.json")))
    problems = []
    if not str(cfg.get("audio_feature_set", "")).startswith("wavlm"):
        problems.append("config.json still describes the old audio features (eGeMAPS) -- run the patched Phase_01")
    if not str(cfg.get("video_backbone", "")).startswith("mtcnn"):
        problems.append("config.json still describes the old video features (DenseNet) -- run the patched Phase_01")
    return problems

def audio_ft_current():
    """STEP K: the fine-tuned audio set must be consistent with config.json (Phase 1 resets config), the files on disk and the frozen Phase 4 features it was built from."""
    cfg = json.load(open(path_in_base("config.json"))); meta = json.load(open(path_in_base("features/audio_ft_meta.json")))
    if meta.get("enabled") is False:
        return ["config.json still points to fine-tuned audio features although MELD_AUDIO_FT=0"] if cfg.get("audio_feature_file") else []
    problems = []
    if cfg.get("audio_feature_file") != "audio_ft_features_{split}.npy" or cfg.get("audio_fusion_feature_dim") != meta.get("dim"):
        problems.append("config.json does not describe the fine-tuned audio features (Phase_01 was re-run?) -- re-run Phase_04b")
    for s in S:
        if not os.path.exists(path_in_base(f"features/audio_ft_features_{s}.npy")) or not os.path.exists(path_in_base(f"features/audio_features_{s}.npy")):
            problems.append(f"missing audio feature file for {s}")
        elif sha1_file(path_in_base(f"features/audio_features_{s}.npy")) != meta.get("frozen_sha1", {}).get(s):
            problems.append(f"frozen audio features ({s}) changed since Phase_04b was built -- re-run Phase_04b")
    return problems

def text_ft_current():
    """STEP E: the fine-tuned text set must be consistent with config.json (Phase 1 resets config), the files on disk and the frozen Phase 3 features it was built from."""
    cfg = json.load(open(path_in_base("config.json"))); meta = json.load(open(path_in_base("features/text_ft_meta.json")))
    if meta.get("enabled") is False:
        return ["config.json still points to fine-tuned text features although MELD_TEXT_FT=0"] if cfg.get("text_feature_file") else []
    problems = []
    if cfg.get("text_feature_file") != "text_ft_features_{split}.npy" or cfg.get("text_fusion_feature_dim") != meta.get("dim"):
        problems.append("config.json does not describe the fine-tuned text features (Phase_01 was re-run?) -- re-run Phase_03b")
    if int(meta.get("settings", {}).get("context_utterances", 0)) < 1:
        problems.append("the stored fine-tuned text features were built WITHOUT dialogue context (Version A needs them) -- re-run Phase_03b")
    want = os.environ.get("MELD_P3B_MODEL", "")
    if want and meta.get("settings", {}).get("ft_model") != want:
        problems.append(f"the stored fine-tuned text features were built with {meta.get('settings', {}).get('ft_model', 'roberta-base')}, not {want} -- re-run Phase_03b")
    for s in S:
        if not os.path.exists(path_in_base(f"features/text_ft_features_{s}.npy")) or not os.path.exists(path_in_base(f"features/text_features_{s}.npy")):
            problems.append(f"missing text feature file for {s}")
        elif sha1_file(path_in_base(f"features/text_features_{s}.npy")) != meta.get("frozen_sha1", {}).get(s):
            problems.append(f"frozen text features ({s}) changed since Phase_03b was built -- re-run Phase_03b")
    return problems

def loss_sweep_current():
    """STEP F: the stored loss decision must belong to the CURRENT features and to a finished sweep (so a stale decision is never applied silently)."""
    sel = json.load(open(path_in_base("results/loss_sweep_selected.json"))); cfg = json.load(open(path_in_base("config.json")))
    now = {k: cfg.get(k) for k in ("audio_feature_set", "audio_model", "video_backbone", "video_hsemotion_model", "text_feature_set", "audio_ft_feature_set") if cfg.get(k) is not None}
    problems = []
    if sel.get("select_metric", "macro_f1") != os.environ.get("MELD_SELECT_METRIC", "macro_f1").strip().lower():
        problems.append("the stored loss decision used a different selection metric -- re-run Phase_09a")
    if sel.get("feature_desc") != now:
        problems.append("loss sweep was made for different features -- re-run Phase_09a")
    if sel.get("enabled"):
        for tag in sel.get("mean_of_two_val_macro_f1", {}):
            if not os.path.exists(path_in_base(f"results/sweeps/{sel.get('sweep_prefix', 'loss')}_{tag}/phase9_val_summary.csv")):
                problems.append(f"sweep results for variant {tag} are missing")
    return problems

CUSTOM_CHECKS = {"audio_ft_current": audio_ft_current, "loss_sweep_current": loss_sweep_current, "text_ft_current": text_ft_current, "phase9_runs_complete": phase9_runs_complete, "features_config_current": features_config_current}

def validate(phase):
    """Returns a list of problems (empty list = phase output is complete and valid)."""
    problems = []
    for rel in phase.get("artifacts", []):
        p = path_in_base(rel)
        if not os.path.exists(p) or os.path.getsize(p) == 0:
            problems.append(f"missing/empty {rel}")
    for rel in phase.get("verify", []):
        p = path_in_base(rel)
        if not os.path.exists(p):
            problems.append(f"missing verification file {rel}")
            continue
        try:
            if json.load(open(p)).get("n_failed", 1) != 0:
                problems.append(f"verification failures recorded in {rel}")
        except Exception as e:
            problems.append(f"unreadable {rel}: {e}")
    if not problems and phase.get("custom"):
        try:
            problems += CUSTOM_CHECKS[phase["custom"]]()
        except Exception as e:
            problems.append(f"custom check failed: {e}")
    return problems

def load_state():
    return json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}

def save_state(state):
    os.makedirs(BASE_DIR, exist_ok=True)
    tmp = STATE_PATH + ".tmp"
    with open(tmp, "w") as f:
        json.dump(state, f, indent=2)
    os.replace(tmp, STATE_PATH)

def sha1_file(path):
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def find_notebook(name, max_depth=3):
    roots = [d for d in ([NOTEBOOK_DIR] if NOTEBOOK_DIR else []) + [BASE_DIR, os.getcwd(), "/kaggle/working"] if d]
    for d in roots:
        if os.path.isfile(os.path.join(d, name)):
            return os.path.join(d, name)
    for top in ([NOTEBOOK_DIR] if NOTEBOOK_DIR else []) + ["/kaggle/input"]:     
        frontier = [(top, 0)] if os.path.isdir(top) else []
        while frontier:
            d, depth = frontier.pop(0)
            try:
                entries = list(os.scandir(d))
            except OSError:
                continue
            for e in entries:
                if e.is_file() and e.name == name:
                    return e.path
            if depth < max_depth:
                frontier += [(e.path, depth + 1) for e in entries if e.is_dir(follow_symlinks=False)]
    raise FileNotFoundError(f"Notebook {name} not found. Put the Phase_XX notebooks in BASE_DIR or set MELD_NOTEBOOK_DIR.")

def restore_from_previous(src):
    """Copy a previous session's working directory (e.g. a Kaggle output attached as a dataset) if BASE_DIR is empty."""
    if not src:
        return
    if os.path.exists(path_in_base("config.json")) or os.path.exists(STATE_PATH):
        print(f"[restore] BASE_DIR already contains a pipeline ({BASE_DIR}); not restoring from {src}")
        return
    if not os.path.isdir(src):
        raise FileNotFoundError(f"MELD_RESTORE_FROM={src} is not a directory")
    os.makedirs(BASE_DIR, exist_ok=True)
    shutil.copytree(src, BASE_DIR, dirs_exist_ok=True)
    print(f"[restore] copied {src} -> {BASE_DIR}")

In [ ]:

RUNNER_SRC = r"""
import sys, nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError
src, dst, cwd = sys.argv[1:4]
nb = nbformat.read(src, as_version=4)
client = NotebookClient(nb, timeout=None, kernel_name="python3", resources={"metadata": {"path": cwd}})
ok = True
try:
    with client.setup_kernel():
        total = len(nb.cells)
        for i, cell in enumerate(nb.cells):
            if cell.cell_type != "code":
                continue
            client.execute_cell(cell, i)
            print(f"[cell {i}/{total - 1} done]", flush=True)
            for o in cell.get("outputs", []):
                if o.get("output_type") == "stream":
                    t = o["text"]
                    print(t if len(t) < 6000 else t[:1500] + "\\n...[truncated]...\\n" + t[-4000:], end="", flush=True)
except CellExecutionError as e:
    ok = False
    print("CELL FAILED:", str(e)[-2500:], flush=True)
finally:
    nbformat.write(nb, dst)
sys.exit(0 if ok else 1)
"""
ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")

def run_notebook(phase, nb_path):
    os.makedirs(EXEC_DIR, exist_ok=True); os.makedirs(LOG_DIR, exist_ok=True)
    runner = os.path.join(BASE_DIR, "_run_notebook.py")
    with open(runner, "w") as f:
        f.write(RUNNER_SRC)
    dst, log = os.path.join(EXEC_DIR, f"{phase['id']}.ipynb"), os.path.join(LOG_DIR, f"{phase['id']}.log")
    proc = subprocess.Popen([sys.executable, runner, nb_path, dst, BASE_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=os.environ.copy())
    try:
        with open(log, "w") as lf:
            for line in proc.stdout:
                line = ANSI.sub("", line)
                print(line, end="", flush=True); lf.write(line)
    except BaseException:
        proc.terminate()
        raise
    rc = proc.wait()
    err = None
    if rc != 0 and os.path.exists(dst):
        try:
            for c in json.load(open(dst))["cells"]:
                for o in c.get("outputs", []):
                    if o.get("output_type") == "error":
                        err = f"{o['ename']}: {o['evalue']}"
        except Exception:
            pass
    return rc, dst, err

In [ ]:
def now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")

def run_pipeline():
    restore_from_previous(RESTORE_FROM)
    ids = [p["id"] for p in PHASES]
    for name, v in (("START_AT", START_AT), ("STOP_AFTER", STOP_AFTER)):
        if v and v not in ids:
            raise ValueError(f"{name}={v} is not one of {ids}")
    for f in FORCE:
        if f not in ids:
            raise ValueError(f"FORCE contains unknown phase {f}")
    start_i, stop_i = (ids.index(START_AT) if START_AT else 0), (ids.index(STOP_AFTER) if STOP_AFTER else len(ids) - 1)
    state, decisions, ran_any, warnings, ran_ids = load_state(), [], False, [], []
    print(f"\n{'PLAN (dry run)' if DRY_RUN else 'PIPELINE'}: {ids[start_i]} .. {ids[stop_i]}\n")

    for i, p in enumerate(PHASES):
        pid = p["id"]
        nb_path = find_notebook(p["nb"])
        sha = sha1_file(nb_path)
        problems, rec = validate(p), state.get(pid)

        if i < start_i:
            if problems:
                raise RuntimeError(f"Cannot start at {START_AT}: earlier phase {pid} is incomplete ({'; '.join(problems[:3])}). Run it first.")
            decisions.append((pid, "valid", "validated, not run (before START_AT)")); print(f"[{pid}] VALID    before START_AT"); continue
        if i > stop_i:
            decisions.append((pid, "pending", "after STOP_AFTER")); print(f"[{pid}] PENDING  after STOP_AFTER"); continue

        reasons = []
        if pid in FORCE:
            reasons.append("forced")
        if CASCADE and any(r not in p.get("independent_of", ()) for r in ran_ids):
            reasons.append("an earlier phase was re-run")
        if problems:
            reasons.append("artifacts incomplete: " + "; ".join(problems[:3]) + (" ..." if len(problems) > 3 else ""))
        if rec and rec.get("notebook_sha1") != sha:
            reasons.append("notebook changed since the last run")
        if rec and rec.get("status") == "failed":
            reasons.append("last run failed")
        if not rec and not ADOPT_EXISTING:
            reasons.append("no run record (adoption disabled)")

        if not reasons:
            if rec is None:
                state[pid] = {"status": "adopted", "notebook": p["nb"], "notebook_sha1": sha, "adopted_at": now()}
                if not DRY_RUN:
                    save_state(state)
                decisions.append((pid, "adopt", "valid artifacts found without a run record")); print(f"[{pid}] ADOPT    valid artifacts found, no run record")
            else:
                decisions.append((pid, "skip", f"completed {rec.get('finished', rec.get('adopted_at', ''))}")); print(f"[{pid}] SKIP     already complete")
            continue

        decisions.append((pid, "run", "; ".join(reasons)))
        print(f"[{pid}] RUN      {'; '.join(reasons)}")
        ran_any = True; ran_ids.append(pid)
        if DRY_RUN:
            continue

        t0 = time.time()
        state[pid] = {"status": "running", "notebook": p["nb"], "notebook_sha1": sha, "started": now()}
        save_state(state)
        rc, executed, err = run_notebook(p, nb_path)
        problems = validate(p) if rc == 0 else []
        secs = round(time.time() - t0, 1)
        if rc != 0 or problems:
            state[pid].update({"status": "failed", "finished": now(), "seconds": secs, "error": err or "; ".join(problems[:5]), "executed_copy": executed})
            save_state(state)
            if p.get("optional"):
                warnings.append(pid)
                decisions[-1] = (pid, "failed", "OPTIONAL phase failed; pipeline continued -- " + str(state[pid]["error"])[:200])
                print(f"[{pid}] FAILED (optional, pipeline continues) after {secs}s: {state[pid]['error']}\n  log: {os.path.join(LOG_DIR, pid + '.log')}\n")
                continue
            raise RuntimeError(f"{pid} FAILED after {secs}s: {state[pid]['error']}\n"
                               f"Executed copy with outputs: {executed}  |  log: {os.path.join(LOG_DIR, pid + '.log')}\n"
                               f"Fix the cause and re-run Main: completed earlier phases are skipped, {pid} is repeated.")
        state[pid].update({"status": "completed", "finished": now(), "seconds": secs, "executed_copy": executed})
        save_state(state)
        print(f"[{pid}] COMPLETED in {secs}s\n")

    print("\nSummary:")
    for pid, d, why in decisions:
        print(f"  {pid}  {d.upper():8s} {why}")
    if not DRY_RUN and stop_i == len(ids) - 1:
        left = [p["id"] for p in PHASES if validate(p) and not p.get("optional")]
        if left:
            raise RuntimeError(f"Pipeline finished but these phases have invalid artifacts: {left}")
        soft = [p["id"] for p in PHASES if p.get("optional") and validate(p)]
        print("\nPIPELINE COMPLETE: final report at results/final_report.md")
        if soft:
            print(f"WARNING: optional phase(s) {soft} have no valid output (see logs/); the thesis results above do not depend on them.")
        else:
            print("Seed-ensemble results: results/ensemble/ensemble_results.json")
    return decisions
print("run_pipeline() defined.")

In [ ]:
decisions = run_pipeline()